# Exploring Spatial Representations of Paired-End Reads Towards the Reconstruction of Intra-Tumor Heterogeneity
## Introduction
Cancer is a disease broadly described as the uncontrollable growth of abnormal cells as a result of (in some part) somatic mutations. These mutations are dynamic; a group of cells which have experienced one mutation may develop, due to the instability of rapid multiplication, another mutation. This instability and mutation produces **intra-tumor heterogeneity**, or the phenomenon of distinct populations within a cell population. We consider the problem of reconstructing the relative frequency of individual populations within a tumor, as well as characterizating their associated mutations. In particular, we look at classifying copy number abberations (CNAs), which delete or amplify a region of a genome. 

In [ ]:
# import relevant libraries and modules
import numpy as np
from src.seqs_and_reads.consts import *
from src.seqs_and_reads.sequencing import paired_end_reads
from src.algorithm.clusters import square_density, only_mutations
import plotly.graph_objects as go 
import plotly.express as px
import matplotlib.pyplot as plt
import pandas as pd
import sklearn.cluster
from plotly.subplots import make_subplots

from IPython.core.display import HTML
HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

### Exploring Populations
To clarify the problem, we provide a visual of the relationships that we are investigating. Let us take a reference genome, labeled in gray, where each base pair can be assigned a reference position along the genome. Observe a mutation occurrence (a deletion) at the set of positions constrained by $(x_1, x_2)$ (the region highlighted in blue). We may also have an insertion, characterized by multiple copies of the same region, which is shown in red. An insertion may not just duplicate a region, but triple it or beyond; we refer to the number of copies a region affected by a CNA mutation has by $m$ (also letting us take $m = 0$ for the deletion case). 

<center><img src="notebook_images/insertion_deletion.png" alt="image of 4 rectangles, representing two reference genomes and their associated CNA (insertion/deletion)" style="width: 600px;"/></center>


At each addition of a set of mutations, a subset of the affected population may accumulate additional mutations, leading to a 'new' population distinct from its parent. This group makes up a portion of the total tumor population with some frequency: a sample population is shown below. We denote by $q_i$ the relative frequency of each population "down the line". For this exploration, we make the assumption that each 'generation' has only one population. 

<center><img src="notebook_images/population_example.png" alt="example of populations inheriting mutations" style="width: 750px;"/></center>

We consider the problem of, from some data, finding each $q_i$, as well as the $m$ (and location, $(a, b)$) of each mutation in the tumor population, with constraint $\sum_{i = 0}^{n} q_i = 1$ (the number of populations, $n$ is also not necessarily known). Our data is the sequencing of this tumor population by paired-end reads, which we describe next. 

### Paired-End Reads
Paired-end sequencing generates 'pairs' of reads of fixed length, but of variable (random) separation. This separation, whose length is constrained to those within the integer pair $(p, q)$, is capped on both ends by our reads, both of length $k$ base pairs. Mapping these base pairs onto a reference genome, we can describe the read by the pair of intervals $[a_1, a_2] \cup [b_1, b_2]$, as labeled below. 

<center><img src="notebook_images/paired-end-example.png" alt="labeled diagram of paired end components" style="width: 750px;"/></center>

Known is $k$, $p$, and $q$, but not the exact separation of the two intervals. Our focus is on the representation of these reads as a coordinate on a plane in $\mathbb{R}^2$. We plot the point $(a_1, b_2)$ to represent a read whose first coordinate is the starting position of the first read, and whose second coordinate is the ending position of the second coordinate. When a genome with no mutations is sequenced and represented in this manner, we should see the following (domain constrained for legibility):

In [ ]:
# generate genome 
genome = np.arange(1000000)
coords = paired_end_reads(genome, 1)
x, y = coords.T.tolist()
df = pd.DataFrame({"x": x, "y": y})
plot_df = df.sample(n = 4999)

# draw lower and upper bound lines
line_x = np.arange(0, 1100000, 100000)
upper = line_x + 2 * READ_LEN + READ_SEP[1]
lower = line_x + 2 * READ_LEN + READ_SEP[0]

fig = go.Figure()
fig.add_trace(go.Scatter(x = plot_df["x"], y = plot_df["y"], 
                         mode = "markers", name = "paired-end reads"))
fig.add_trace(go.Scatter(x = line_x, y = upper, mode = 'lines', 
                         line = dict(color = "#c26e63"), name = "upper limit"))
fig.add_trace(go.Scatter(x = line_x, y = lower, mode = 'lines', 
                         line = dict(color = "#dbafa9"), name = "lower limit"))

fig.update_layout(
    autosize = False,
    width = 1000, 
    height = 800,
    margin = dict(
        l = 20,
        r = 20, 
        b = 20, 
        t = 20,
    )
)

fig.update_xaxes(range = [50000, 90000], constrain = "domain")
fig.update_yaxes(range = [53000, 90000])

fig.show()

Notice that all points are bounded by the region $x + 2k + p$ and $x + 2k + q$, as the separation between $x$ and $y$ in a coordinate pair is constrained by its minimum and maximum separation length. We'll call this bounded region the 'main region.'

Now consider that a deletion occurs in the region (70000, 100000) below, and observe reads for this deletion.

In [ ]:
# delete region and plot reads
deletion_range = [70000, 100000]
del_genome = np.concatenate((np.arange(deletion_range[0]), np.arange(deletion_range[1], 1000000)))

del_coords = paired_end_reads(del_genome, 1)
x, y = del_coords.T.tolist()
ddf = pd.DataFrame({"x": x, "y": y})
plot_ddf = ddf.sample(n = 4999)

# draw lower and upper bound lines
line_x = np.arange(0, 1100000, 100000)
upper = line_x + 2 * READ_LEN + READ_SEP[1]
lower = line_x + 2 * READ_LEN + READ_SEP[0]

fig = go.Figure()
fig.add_trace(go.Scatter(x = plot_ddf["x"], y = plot_ddf["y"], 
                         mode = "markers", name = "paired-end reads"))
fig.add_trace(go.Scatter(x = line_x, y = upper, mode = 'lines', 
                         line = dict(color = "#c26e63"), name = "upper limit"))
fig.add_trace(go.Scatter(x = line_x, y = lower, mode = 'lines', 
                         line = dict(color = "#dbafa9"), name = "lower limit"))

fig.update_layout(
    autosize = False,
    width = 1000, 
    height = 800,
    margin = dict(
        l = 20,
        r = 20, 
        b = 20, 
        t = 20,
    )
)

fig.update_xaxes(range = [60000, 110000], constrain = "domain")
fig.update_yaxes(range = [53000, 120000])

fig.show()

Because the length of the deletion can be assumed to be sufficiently larger than $q$, what we will find is that our paired end is mapped to a region outside of the original boundaries: above the main region, because it appears as though our separation is much larger than it actually is. When we read over the whole genome, we also see a disruption in the main region in the lack of points over the mutation, as is natural. 

In [ ]:
# insertion region and plot reads
ins_range = [500000, 550000]
insertion = np.tile(np.arange(ins_range[0], ins_range[1]), 3)
ins_genome = np.concatenate((np.arange(ins_range[0]), insertion, np.arange(ins_range[1], 1000000)))

i_coords = paired_end_reads(ins_genome, 1)
x, y = i_coords.T.tolist()
idf = pd.DataFrame({"x": x, "y": y})
plot_idf = idf.sample(n = 4999)

# draw lower and upper bound lines
line_x = np.arange(0, 1100000, 100000)
upper = line_x + 2 * READ_LEN + READ_SEP[1]
lower = line_x + 2 * READ_LEN + READ_SEP[0]

fig = go.Figure()
fig.add_trace(go.Scatter(x = plot_idf["x"], y = plot_idf["y"], 
                         mode = "markers", name = "paired-end reads"))
fig.add_trace(go.Scatter(x = line_x, y = upper, mode = 'lines', 
                         line = dict(color = "#c26e63"), name = "upper limit"))
fig.add_trace(go.Scatter(x = line_x, y = lower, mode = 'lines', 
                         line = dict(color = "#dbafa9"), name = "lower limit"))

fig.update_layout(
    autosize = False,
    width = 1000, 
    height = 800,
    margin = dict(
        l = 20,
        r = 20, 
        b = 20, 
        t = 20,
    )
)

fig.update_xaxes(range = [500000, 560000], constrain = "domain")
fig.update_yaxes(range = [490000, 550000])

fig.show()

We also see displacements and main region density disturbances when an insertion of multiplicity $m = 3$ at (500000, 550000) occurs. We will again see a difference in the main region: this time, that the portion of the main region associated with reads in the insertion has much higher density than unmutated regions. In fact, the ratio of the density of the points in this region to the base density of populations containing the mutation should be exactly $m$, or 3 in this case.

These relationships are complicated by the fact that not every population may have an insertion, or deletion, at the specified region; Taking as an example a mixture of our reference genome and the deletion genome, we may have the following:

In [ ]:
# generate genome 
genome = np.arange(1000000)
coords = paired_end_reads(genome, 0.7)
x, y = coords.T.tolist()
df = pd.DataFrame({"x": x, "y": y})
plot_df = df.sample(n = int(4999 * 0.7) )

deletion_range = [70000, 100000]
del_genome = np.concatenate((np.arange(deletion_range[0]), np.arange(deletion_range[1], 1000000)))

del_coords = paired_end_reads(del_genome, 0.3)
x, y = del_coords.T.tolist()
ddf = pd.DataFrame({"x": x, "y": y})
plot_ddf = ddf.sample(n = int(4999 * 0.3))

# draw lower and upper bound lines
line_x = np.arange(0, 1100000, 100000)
upper = line_x + 2 * READ_LEN + READ_SEP[1]
lower = line_x + 2 * READ_LEN + READ_SEP[0]

fig = go.Figure()
fig.add_trace(go.Scatter(x = plot_ddf["x"], y = plot_ddf["y"], 
                         mode = "markers", name = "deletion population"))
fig.add_trace(go.Scatter(x = line_x, y = upper, mode = 'lines', 
                         line = dict(color = "#c26e63"), name = "upper limit"))
fig.add_trace(go.Scatter(x = line_x, y = lower, mode = 'lines', 
                         line = dict(color = "#dbafa9"), name = "lower limit"))
fig.add_trace(go.Scatter(x = plot_df["x"], y = plot_df["y"], 
                         mode = "markers", name = "reference reads", 
                         marker = dict(color = "gray")))

fig.update_layout(
    autosize = False,
    width = 1000, 
    height = 800,
    margin = dict(
        l = 20,
        r = 20, 
        b = 20, 
        t = 20,
    )
)

fig.update_xaxes(range = [60000, 110000], constrain = "domain")
fig.update_yaxes(range = [53000, 120000])

fig.show()

This mixture of data obfuscates the relationships that may be utilized to reconstruct the location of mutation regions from the spatial representation, but still provides some important data. Using these relationships, we hope to be able to take unlabeled coordinate data and find the locations and multiplicities of their CNAs, as well as the relative frequencies of these mutations within the population.

## Solving the Problem
### Identifying Regions of Mutation
When we read in and plot our data, we find the following:

In [ ]:
df = pd.read_csv('output/try1_coords.csv')[["left read", "right read"]]

plot_df = df.sample(n = 4999)

# draw lower and upper bound lines
line_x = np.arange(0, 1100000, 100000)
upper = line_x + 2 * READ_LEN + READ_SEP[1]
lower = line_x + 2 * READ_LEN + READ_SEP[0]

fig = go.Figure()
fig.add_trace(go.Scatter(x = plot_df["left read"], y = plot_df["right read"], 
                         mode = "markers", name = "paired-end reads"))
fig.add_trace(go.Scatter(x = line_x, y = upper, mode = 'lines', 
                         line = dict(color = "#c26e63"), name = "upper limit"))
fig.add_trace(go.Scatter(x = line_x, y = lower, mode = 'lines', 
                         line = dict(color = "#dbafa9"), name = "lower limit"))

fig.update_layout(
    autosize = False,
    width = 1000, 
    height = 800,
    margin = dict(
        l = 20,
        r = 20, 
        b = 20, 
        t = 20,
    )
)

fig.update_xaxes(range = [200000, 600000], constrain = "domain")
fig.update_yaxes(range = [200000, 600000])

fig.show()

As expected, we see displaced clusters of points corresponding to different mutations. How can we characterize these clusters of points so that they are separated from each other? To do this, we first delete the main region (whose bounds are defined by the previously identified upper and lower limits), and cluster the displacement regions with the [DBSCAN Algorithm](https://en.wikipedia.org/wiki/DBSCAN). Each cluster is provided its own color below. 

In [ ]:
# find clusters
mut_df = only_mutations(df)
X = mut_df.to_numpy()
clustering = sklearn.cluster.DBSCAN(eps = 100, min_samples = 4).fit(X)
mut_df["cluster"] = clustering.labels_

fig = go.Figure(data=go.Scatter(x=mut_df['left read'],
                                y=mut_df['right read'],
                                mode='markers',
                                marker_color=mut_df['cluster'],
                                )) # hover text goes here

fig.show()

By observing if a cluster is above or below the main region, we know if it is an insertion or deletion region: we also have some information about the location of each mutation based on its largest and smallest points. The largest $x$ value in a deletion cluster should roughly correspond to the starting position of the mutation, and the smallest $y$ value its end position; in an insertion, we flip these starting and ending positions as we generally expect that $y < x$ (the starting position of the mutation now being the smallest $y$, and the end position the largest $x$). We collect this information into a table of cluster data, which we start below. A 'fuzzy left' and 'fuzzy right' column indicate the expected position of our CNA regions. 

In [ ]:
# find fuzzy borders of clusters
cluster_df = mut_df.groupby('cluster').agg({'left read': ['min', 'max'], 
                                            'right read': ['min', 'max']})
cluster_df.columns = ['_'.join(col) for col in cluster_df.columns]
cluster_df.reset_index(inplace = True)
cluster_df.columns = cluster_df.columns.str.replace(' ', '_')

cluster_df['fuzzy_left'] = cluster_df[['right_read_min', 'left_read_max']].min(axis = 1)
cluster_df['fuzzy_right'] = cluster_df[['right_read_min', 'left_read_max']].max(axis = 1)
cluster_df.head()

### Generating Potential Solutions
What information do we know from our clusters? Taking a step back, we denote by $p_0$ the relative frequency of populations that do not have a certain mutation, and by $p_1$ the relative frequency of populations that do. When normalizing the density of points, we naturally want $p_0 + p_1 = 1$. 

Introducing this $p_0$, $p_1$ relation allows us to introduce an important relationship within our clusters *for insertions*. Take regions $A$, $B$ for the mutation described below. The relationship between the density $D_A$ of points in region $A$ and the density $D_B$ of points in region $B$ is described by the system of equations

$$\begin{cases}
    (m - 1)p_1 = D_A \\
    mp_1 + p_0 = D_B
\end{cases}
$$

Justification of this relationship is provided by the image below. 

<center><img src="notebook_images/linear_paired_end.png" alt="explanation of density relationships" style="width: 750px;"/></center>

Calculating the relationship between the density of regions $A$ and $B$, we obtain an updated table of cluster data. we also take the opportunity to classify each mutation as an insertion or deletion. density $C$ is a normalization factor. 

In [ ]:
# identify centroid from which density is calculated
# mutation centriod
cluster_df['centroid_A'] = cluster_df.apply(lambda row: ((row.left_read_min + row.left_read_max)//2 + READ_LEN, (row.right_read_min + row.right_read_max)//2 - READ_LEN), axis = 1)
# full multiplicity centroid
cluster_df['centroid_B'] = cluster_df.apply(lambda row: ((row.fuzzy_left + row.fuzzy_right)//2 - sum(READ_SEP)//4 - READ_LEN, (row.fuzzy_left + row.fuzzy_right + sum(READ_SEP)//2)//2 + READ_LEN), axis = 1)
# normalization centroid
cluster_df['centroid_C'] = cluster_df.apply(lambda row: (row.fuzzy_left - sum(READ_SEP)//4 - READ_LEN, row.fuzzy_left + sum(READ_SEP)//4 + READ_LEN), axis = 1)

# calculate densities
A_density = []
B_density = []
C_density = []

for index, row in cluster_df.iterrows():
    a = square_density(df, row['centroid_A'], READ_LEN * 2)
    b = square_density(df, row['centroid_B'], READ_LEN * 2)
    c = square_density(df, row['centroid_C'], READ_LEN * 2)
    A_density.append(a)
    B_density.append(b)
    C_density.append(c)

cluster_df["density_A"] = A_density 
cluster_df["density_B"] = B_density
cluster_df["density_C"] = C_density

cluster_df["mtype"] = cluster_df.apply(lambda row: "deletion" if row.left_read_min + 2 * READ_LEN + READ_SEP[1] < row.right_read_min else "insertion", axis = 1)
cluster_df.head()

For each guess of the multiplicity $m$, we can solve the system of equations. The value of $m$ can be reasonably constrained by some upper limit (we take 7) which generates 6 guesses of the triple $(m, p_0, p_1)$ for each mutation. Within these guesses should contain the ground truth. 

In [ ]:
# find base density for normalization
#TODO: pick like,,, 10 points to average this into something reasonable
normal_found = False
LENGTH = df['right read'].max()
while not normal_found:
    # chooose a point at random from deletion buffer
    sdel = np.random.randint(MUT_LENGTH[0], MUT_LENGTH[1])
    spos0 = np.random.randint(READ_SEP[1], LENGTH - sdel - READ_SEP[1])
    centroid = (spos0 - sum(READ_SEP)//4 - READ_LEN, spos0 + sdel + sum(READ_SEP)//4 + READ_LEN)
    # check centroid
    d0 = square_density(df, centroid, READ_LEN * 2)
    if d0 == 0:
        # check that this isn't an insertion region either:
        centroid = (spos0 + sdel - sum(READ_SEP)//4 - READ_LEN, spos0 + sum(READ_SEP)//4 + READ_LEN)
        d1 = square_density(df, centroid, READ_LEN * 2) 
        if d1 == 0:
            normal_found = True
            centroid = (spos0 + sdel//2, spos0 + sdel//2 + sum(READ_SEP)//2 + 2 * READ_LEN)
            normed_density = square_density(df, centroid, READ_LEN * 2)
        else:
            print(f"not quite: d1 had density {d1} with centroid {centroid}")
    else:
        print(f"not quite: d0 had density {d0} with centroid {centroid}")

# return probable densities for each as pairs of choices of m
# triples of (m, q0, q1)
def A_based_calculation(df, normed_density):
    n = df.shape[0]
    stored = []
    for i in range(n):
        possibilities = []
        dA = df.loc[i]["density_A"]
        mtype = df.loc[i]["mtype"]
        if mtype == "deletion":
            possibilities.append((0, 1 - dA/normed_density, 
                                  dA / normed_density))
            stored.append(possibilities)
        if mtype == "insertion":
            for j in range(2, 7):
                q1 = dA / (j - 1) / normed_density
                q0 = 1 - q1
                possibilities.append((j, q0, q1))
            stored.append(possibilities)
    return stored

A_calc = A_based_calculation(cluster_df, normed_density)
cluster_df["A_preds"] = A_calc
cluster_df.head()

### Reducing the Solution Space (SECTION INCOMPLETE)
Of course, some guesses are better than others. One of original run of guesses stated that 126% of the population did not have a particular mutation, a nonsensical value that can immediately be rejected as a solution. We can utilize additional data to cut down on the space of possible solutions - deletions, whose certain multiplicty give us concrete information about the population mixture. Beyond these 'common sense' measures, there are some other constraints on the solution space that we can place: 

### Evaluating Guesses By Assumptions of Population-Mixture Size (SECTION INCOMPLETE)
We preface this discussion by emphasizing that theoretical limits exist towards the solution of this problem; consider, for example, the two-mixture population with reference population frequency $q_0 = 0.68$. This mixture, experiencing an insertion of multiplicity $m = 2$ in region $(500000, 550000)$, is indistinguishable from the case in which an insertion of multiplicity $m = 3$ and $q_0 = 0.84$. We show these plots below. 

In [ ]:
# reference genome
ref = np.arange(1000000)
ref_coords = paired_end_reads(ref, 1)
xr, yr = ref_coords.T.tolist()
ref_df = pd.DataFrame({"x": xr, "y": yr})

# m = 3
ins_range = [500000, 550000]
insertion1 = np.tile(np.arange(ins_range[0], ins_range[1]), 3)
ins_genome1 = np.concatenate((np.arange(ins_range[0]), insertion, np.arange(ins_range[1], 1000000)))

i1_coords = paired_end_reads(ins_genome, 0.16)
x1, y1 = i_coords.T.tolist()
i1df = pd.DataFrame({"x": x1, "y": y1})
plot1_idf = i1df.sample(n = int(20000 * 0.16))
ref1_df = ref_df.sample(n = int(20000 * 0.84))

# m = 2
insertion2 = np.tile(np.arange(ins_range[0], ins_range[1]), 2)
ins_genome2 = np.concatenate((np.arange(ins_range[0]), insertion2, np.arange(ins_range[1], 1000000)))
i2_coords = paired_end_reads(ins_genome, 1)
x2, y2 = i2_coords.T.tolist()
i2df = pd.DataFrame({"x": x2, "y": y2})
plot2_idf = i2df.sample(n = int(10000 * 0.32))
ref2_df = ref_df.sample(n = int(10000 * 0.68))

# draw lower and upper bound lines
line_x = np.arange(0, 1100000, 100000)
upper = line_x + 2 * READ_LEN + READ_SEP[1]
lower = line_x + 2 * READ_LEN + READ_SEP[0]

fig = make_subplots(rows = 1, cols = 2, subplot_titles = ("m = 3", "m = 2"))
fig.add_trace(go.Scatter(x = plot1_idf["x"], y = plot1_idf["y"], 
                         mode = "markers", name = "insertion pop.", marker = dict(color = "#3283c9")),
                         row = 1, col = 1)
fig.add_trace(go.Scatter(x = ref1_df["x"], y = ref1_df["y"], 
                         mode = "markers", name = "reference pop.", marker = dict(color = "#868786")),
                         row = 1, col = 1)
fig.add_trace(go.Scatter(x = line_x, y = upper, mode = 'lines', 
                         line = dict(color = "#c26e63"), name = "upper limit"),
                         row = 1, col = 1)
fig.add_trace(go.Scatter(x = line_x, y = lower, mode = 'lines', 
                         line = dict(color = "#dbafa9"), name = "lower limit"),
                         row = 1, col = 1)
fig.add_trace(go.Scatter(x = line_x, y = upper, mode = 'lines', 
                         line = dict(color = "#c26e63"), name = "upper limit"),
                         row = 1, col = 2)
fig.add_trace(go.Scatter(x = line_x, y = lower, mode = 'lines', 
                         line = dict(color = "#dbafa9"), name = "lower limit"),
                         row = 1, col = 2)
fig.add_trace(go.Scatter(x = plot2_idf["x"], y = plot2_idf["y"], 
                         mode = "markers", name = "insertion pop.", marker = dict(color = "#3283c9")),
                         row = 1, col = 2)
fig.add_trace(go.Scatter(x = ref2_df["x"], y = ref2_df["y"], 
                         mode = "markers", name = "reference pop.", marker = dict(color = "#868786")),
                         row = 1, col = 2)

fig.update_layout(
    autosize = False,
    width = 1800, 
    height = 800,
    margin = dict(
        l = 20,
        r = 20, 
        b = 20, 
        t = 20,
    ),
    showlegend = False
)

fig.update_xaxes(range = [500000, 560000], constrain = "domain")
fig.update_yaxes(range = [490000, 550000])

fig.show()

Having identified some theoretical caps, there are of course still constructions more likely than others. We emphasize two optimization assumptions, *in order*:

1. A guess of fewer populations is more likely than a guess of additional populations. 
2. 



For validation, we provide the relative population frequencies and mutations from which our data was constructed: for a 4 population mixture, we had

$q0 = 0.3$

$q1 = 0.2$
- insertion $m = 3$ at $[470744, 478852]$

$q2 = 0.2$ 
- insertion $m = 2$ at $[429222, 438159]$
- deletion $m = 0$ at $[314485, 320869]$

$q3 = 0.3$ 
- deletion $m = 0$ at $[521169, 529053]$


